In [1]:
from transformers import DataCollatorForLanguageModeling, BertConfig, BertForPreTraining, TrainingArguments, Trainer

In [2]:
from datasets import load_dataset

In [3]:
dataset= load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [4]:
train_texts= dataset["train"]["text"]

In [5]:
train_texts[1]

' = Valkyria Chronicles III = \n'

In [6]:
sentences= [ s for s in train_texts if len(s.strip()) > 0]

In [7]:
import random

In [8]:
def create_sentence_pairs(sentences):
  pairs= []
  for i in range(len(sentences) - 1):
    if random.random() < 0.5:
      pairs.append({
          "sentence_a": sentences[i],
          "sentence_b": sentences[i+1],
          "label": 1
      })
  else:
    random_choice= random.choice(sentences)
    pairs.append({
        "sentence_a": sentences[i],
        "sentence_b": random_choice,
        "label": 0
    })
  return pairs

In [9]:
pairs= create_sentence_pairs(sentences)

In [10]:
from datasets import Dataset
dataset= Dataset.from_list(pairs)

In [11]:
type(dataset)

datasets.arrow_dataset.Dataset

In [12]:
from transformers import BertTokenizerFast

In [13]:
tokenizer= BertTokenizerFast.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [14]:
def tokenize(example):
  return tokenizer(
      example["sentence_a"],
      example["sentence_b"],
      truncation= True,
      padding= "max_length",
      max_length= 128
  )

In [15]:
tokenized_data = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/582459 [00:00<?, ? examples/s]

In [16]:
tokenized_data[1].keys()

dict_keys(['sentence_a', 'sentence_b', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [17]:
tokenized_data= tokenized_data.rename_column("label", "next_sentence_label")

In [18]:
tokenized_data= tokenized_data.remove_columns(["sentence_a", "sentence_b"])

In [19]:
tokenized_data[1].keys()

dict_keys(['next_sentence_label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [20]:
tokenized_data.set_format("torch")

In [21]:
from transformers import DataCollatorForLanguageModeling, BertConfig, BertForPreTraining, TrainingArguments, Trainer

In [22]:
data_collator= DataCollatorForLanguageModeling(
    tokenizer= tokenizer,
    mlm= True,
    mlm_probability= 0.15
)

In [23]:
data_collator

DataCollatorForLanguageModeling(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_ten

In [24]:
config= BertConfig(
    vocab_size= tokenizer.vocab_size,
    hidden_size= 256,
    num_hidden_layers= 4,
    num_attention_heads= 4,
    intermediate_size= 1028,
    max_position_embeddings= 128
)

In [25]:
model= BertForPreTraining(config)

In [26]:
training_args= TrainingArguments(
    output_dir= "./bert-mlm-nsp",
    per_device_train_batch_size= 16,
    num_train_epochs= 3,
    logging_steps= 100,
    save_steps= 1000,
    learning_rate= 5e-4,
    weight_decay= 0.01,
    warmup_steps= 500,
    fp16= True
)

In [27]:
trainer= Trainer(
    model= model,
    args= training_args,
    train_dataset= tokenized_data,
    data_collator= data_collator
)

In [28]:
trainer.train()

Step,Training Loss
100,10.021063
200,7.775297
300,7.321027
400,7.145832
500,7.104490
600,7.017231
700,6.951080
800,6.952220
900,6.876318
1000,6.816349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=109212, training_loss=3.8194009385973273, metrics={'train_runtime': 5640.9298, 'train_samples_per_second': 309.768, 'train_steps_per_second': 19.361, 'total_flos': 4470008885978112.0, 'train_loss': 3.8194009385973273, 'epoch': 3.0})

In [29]:
trainer.save_model("./mini-bert-epoch-3")
tokenizer.save_pretrained("./mini-bert-epoch-3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mini-bert-epoch-3/tokenizer_config.json',
 './mini-bert-epoch-3/tokenizer.json')

In [31]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [33]:
model.push_to_hub("Thamo31/MiniEmbedding")
tokenizer.push_to_hub("Thamo31/MiniEmbedding")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...vieuls2/model.safetensors:   1%|1         |  555kB / 44.7MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Thamo31/MiniEmbedding/commit/37c6d991e0ca6767e110207a8067d33fbd16499a', commit_message='Upload tokenizer', commit_description='', oid='37c6d991e0ca6767e110207a8067d33fbd16499a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Thamo31/MiniEmbedding', endpoint='https://huggingface.co', repo_type='model', repo_id='Thamo31/MiniEmbedding'), pr_revision=None, pr_num=None)